# Stellar-Age BNN: Dataset Diagnosis & Prep

Interactive companion to `prepare_dataset.py`. It diagnoses the issues in the existing APOGEE DR17 catalog and shows the fix **without adding new stars**:

1. **Cap** unphysical ages (> 13.8 Gyr) at 14 Gyr instead of dropping them.
2. **Engineer** `[C/N] = C_FE - N_FE` (the strongest spectroscopic age indicator for giants).
3. **Normalize** using train-only statistics (no leakage).
4. **Re-balance** via inverse-frequency `train_weight` per logAge bin.

The prep steps are imported from `prepare_dataset.py`; this notebook adds the visual before/after.

In [65]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repo that holds prepare_dataset.py, then chdir into it so the
# relative data paths in prepare_dataset (./train_data/...) resolve correctly.
# Note: sys.path / chdir do NOT expand '~' on their own -> use expanduser.
CANDIDATES = [
    '~/scr_mk27/bingo-modern',
    '~/code/bingo-modern',
    '.',                       # already inside the repo
]
repo = next((p for p in map(os.path.expanduser, CANDIDATES)
             if os.path.isfile(os.path.join(p, 'prepare_dataset.py'))), None)
if repo is None:
    raise FileNotFoundError(
        'Could not find prepare_dataset.py. Add its directory to CANDIDATES above.')
repo = os.path.abspath(repo)
os.chdir(repo)
sys.path.insert(0, repo)
print('repo:', repo)
import prepare_dataset as prep

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (7, 4)
TARGET, TARGET_ERR = prep.TARGET, prep.TARGET_ERR

: 

## 1. Load the raw catalog

In [66]:
import pandas as pd
data = pd.read_parquet('~/scr_mk27/bulge-ages-and-orbits/data/merged_with_ages.parquet')

In [67]:
data['EvoState'].value_counts()

In [68]:
data2 = data[data['EvoState'] == 2.0]

In [69]:
list(data2.columns)

In [70]:
# Final (derived) columns. Age -> log10(age); [X/H] -> [X/Fe] = [X/H] - [Fe/H].
# The actual derivation + error propagation happens where data_filt is built.
COLUMNS = ['sdss_id', 'log_age', 'e_log_age', 'raw_teff', 'raw_e_teff', 'raw_logg', 'raw_e_logg',
           'raw_fe_h', 'raw_e_fe_h', 'mg_fe', 'e_mg_fe', 'n_fe', 'e_n_fe', 'c_fe', 'e_c_fe']

In [71]:
# Discard all rows where any column ending with '_warn' is nonzero and print the number discarded
warn_cols = [col for col in data2.columns if 'warn' in col]
mask = (data2[warn_cols] == 0).all(axis=1)
n_discarded = (~mask).sum()
print(f"Number of rows discarded due to nonzero *_warn: {n_discarded}")
data2 = data2[mask].copy()

In [72]:
prep.BASE_FEATURES

In [73]:
data_filt = pd.DataFrame({
    'sdss_id':    data2['sdss_id'].values,
    # log10(age). Percentiles are preserved under the monotonic log, so the symmetric
    # 1-sigma is the half-width of the central 68% interval computed in log space.
    'log_age':    np.log10(data2['age_Dnu']).values,
    'e_log_age':  ((np.log10(data2['age_84_Dnu']) - np.log10(data2['age_16_Dnu'])) / 2).values,
    'teff':   data2['raw_teff'].values,
    'e_teff': data2['raw_e_teff'].values,
    'logg':   data2['raw_logg'].values,
    'e_logg': data2['raw_e_logg'].values,
    'fe_h':   data2['raw_fe_h'].values,
    'e_fe_h': data2['raw_e_fe_h'].values,
    # C, N, Mg from the NATIVE ASPCAP/Astra model-atmosphere labels ([X/M]) instead of
    # reconstructing [X/Fe] = [X/H] - [Fe/H]. These are fit relative to the overall
    # metallicity, so the Teff/logg/continuum systematics shared with [Fe/H] are already
    # folded out -- no error-in-quadrature inflation, and the native errors are used as-is.
    # Since [M/H] ~= [Fe/H], [X/Fe] ~= [X/M]; the small offset is absorbed by the train-only
    # normalization downstream.
    #   c_fe  <- [C/M]     (raw_c_m_atm)
    #   n_fe  <- [N/M]     (raw_n_m_atm)
    #   mg_fe <- [alpha/M] (raw_alpha_m_atm): no native [Mg/M] exists; Mg dominates the
    #            alpha tracer, so [alpha/M] is the standard proxy for [Mg/Fe].
    'mg_fe':   data2['raw_alpha_m_atm'].values,
    'e_mg_fe': data2['raw_e_alpha_m_atm'].values,
    'n_fe':    data2['raw_n_m_atm'].values,
    'e_n_fe':  data2['raw_e_n_m_atm'].values,
    'c_fe':    data2['raw_c_m_atm'].values,
    'e_c_fe':  data2['raw_e_c_m_atm'].values,
})

Ask Andy Casey about the uncertainties...

In [74]:
data_filt.head()

In [75]:
data_filt.to_csv('data_filt_He.csv')